<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/_banners/opim5512_banner.svg" alt="OPIM 5512 banner" width="100%">

# Lab 4 · Your Model in Production, with SHAP
### Train it in Colab. Ship it to Google Cloud. Explain every prediction it makes.

*Solo lab · about 2 hours · Colab · Dr. Wanik's cleaned Hartford cars, then your own Google Cloud bucket*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module3/Lab4_Model_In_Production.ipynb)

Since Lab 3 your pipeline scrapes cars, extracts them with **your** regex, and builds a clean master table every hour.
Tonight you put a **model you tuned yourself** into that pipeline, and make it **explain every prediction** with SHAP.

| Part | Time | What you do |
|---|---|---|
| **0. Before you start** | 5 min | Get `predict-shap` deploying (Sync fork, Run workflow, or Cloud Shell) while you work |
| **1. Load** | 10 min | Dr. Wanik's cleaned `listings_master.csv` (267 Hartford cars, after the A06 regex fix) |
| **2. Clean** | 10 min | The last mess: `Ford` vs `ford`, `chevy` vs `chevrolet`, placeholder mileage |
| **3. Baseline** | 10 min | The number to beat: a plain decision tree (the one your pipeline already trains every hour) |
| **4. Tune** | 25 min | Guess the winner, then tune a decision tree, a random forest and KNN. **You** pick the champion |
| **5. Explain** | 20 min | SHAP on your champion, then **price a car you know** |
| **6. Ship** | 20 min | Save the model, upload it to your bucket, and watch `predict-shap` use it |
| **7. Read production** | 15 min | The predictions **and** SHAP values your pipeline saved, and the ads behind them |
| **8. Wrap up** | 5 min | Four short answers, submit |

**First: File → Save a copy in Drive.** This notebook opens from GitHub, so your edits disappear when you close the tab unless you work in your own copy.

**Stuck?** Ask Claude: paste the cell, the full error, and what you expected.

## The big picture
Your pipeline already scrapes (:00), extracts (:10 regex, :12 Gemini), combines (:15) and trains a basic tree (:20).
Tonight adds one more hourly step, **:25 · predict-shap**: it loads **your** model, predicts every new car, and saves
**why** (SHAP) next to **what** (the prediction).

<img src="https://raw.githubusercontent.com/drdave-teaching/OPIM5512-labs/master/Module3/img/OPIM5512_M3_Pipeline_Overview.png" alt="How your pipeline works" width="100%">

# Part 0 · Before you start: get `predict-shap` running (5 min)

Dr. Wanik added a new Cloud Function, **`predict-shap`**, to the course repo. It takes about 3 minutes to deploy, so
start it **now** and let it finish while you work. Open **your** myscrapers repo on GitHub and find your case:

| What you see on your repo's Code tab | Your case |
|---|---|
| **"This branch is N commits behind drdave-teaching/myscrapers"** | **A**: Sync fork |
| "forked from drdave-teaching/myscrapers", **0 behind**, and a `cloud_function/predict-shap` folder | **B**: run the workflow once |
| **No** "forked from drdave-teaching/myscrapers" under the repo name (you imported, cloned, or copied it) | **C**: Cloud Shell |

**Case A: Sync fork** (the same steps as the Sync-fork video)
1. Click **Sync fork**. GitHub offers two buttons. **Click "Update branch."**
   **Never click "Discard commits":** the commits it would discard are **your A06 regex fix**, and "Contact
   Information" would come back. "Update branch" keeps your fix **and** adds `predict-shap` on top.
2. Afterwards your repo says it is **ahead** (your A06 work) and **0 behind**. Good.
3. **Actions** tab: **Deploy predict-shap** starts by itself. No run after a minute? Do Case B.

**Case B: run the workflow once.** You forked after `predict-shap` was added, so the files are there but it was never
deployed. **Actions** tab → **Deploy predict-shap** (left list) → **Run workflow** → **Run workflow**.

**Case C: no fork? Deploy it from Cloud Shell.** Your repo is not linked to the course repo, so Sync fork can never
bring you updates. This block deploys `predict-shap` straight from Dr. Wanik's repo, with exactly the settings the
GitHub workflow uses. Open **Cloud Shell**, change the first two lines, and paste the whole block:
```bash
PROJECT_ID="YOUR-PROJECT-ID"
BUCKET_NAME="YOUR-PROJECT-ID"
REGION="us-central1"
gcloud config set project "$PROJECT_ID"
RUNTIME_SA="cf-runtime@${PROJECT_ID}.iam.gserviceaccount.com"

mkdir -p ~/predict-shap && cd ~/predict-shap
curl -sSLO https://raw.githubusercontent.com/drdave-teaching/myscrapers/main/cloud_function/predict-shap/main.py
curl -sSLO https://raw.githubusercontent.com/drdave-teaching/myscrapers/main/cloud_function/predict-shap/requirements.txt

gcloud functions deploy predict-shap --gen2 --region="$REGION" --runtime=python312 --source=.   --entry-point=predict_shap_http --trigger-http --no-allow-unauthenticated   --timeout=900 --memory=2Gi --service-account="$RUNTIME_SA"   --set-env-vars="PROJECT_ID=$PROJECT_ID,GCS_BUCKET=$BUCKET_NAME,DATA_KEY=structured/datasets/listings_master.csv,OUTPUT_PREFIX=structured/preds_lab4,MODEL_PREFIX=models"

gcloud functions add-invoker-policy-binding predict-shap --gen2 --region="$REGION"   --member="serviceAccount:$RUNTIME_SA"
URL=$(gcloud functions describe predict-shap --gen2 --region="$REGION" --format='value(serviceConfig.uri)')
gcloud scheduler jobs create http predict-shap-hourly --location="$REGION"   --schedule="25 * * * *" --time-zone="America/New_York" --uri="$URL" --http-method=POST   --message-body='{"dry_run":false}' --attempt-deadline=900s   --oidc-service-account-email="$RUNTIME_SA" --oidc-token-audience="$URL"
```
The deploy takes about 3 minutes. If GCP asks to enable an API, type `y`. If the last line says the job already
exists, you are fine. In Case C, when Dr. Wanik updates `predict-shap` later, paste this block again (and change the
last command's `create` to `update`).

Until you upload a model, `predict-shap` politely does nothing (it reports `"status": "no_model"`).

# Part 1 · Load (10 min)

## 1a. Match production's library versions
Your model has to load inside a Google Cloud Function later, and a model saved with one version of scikit-learn may not
load in another. So we install the **exact** versions `predict-shap` uses. (If Colab asks you to **restart the
session**, click it, then run this notebook again from the top.)

In [ ]:
!pip install -q scikit-learn==1.6.1 shap==0.48.0

import sklearn
import shap
print("scikit-learn", sklearn.__version__, "| shap", shap.__version__, "(production uses 1.6.1 and 0.48.0)")

## 1b. Pick your data
- **Start here (everyone, tonight):** leave `USE_MY_OWN_DATA = False`. You get the real `listings_master.csv` from
  Dr. Wanik's pipeline **after** the A06 fix: 267 Hartford cars, make and model pulled by the Lab 3 regex. No sign-in.
  Everyone trains on the same cars, so everyone's numbers are comparable.
- **Your own cars (later, for a custom model):** set `USE_MY_OWN_DATA = True` and re-run from here. You need your A06
  regex fix shipped (otherwise half your makes say `contact`) and ideally 100+ cars. The next cell checks both for you.

Type your project ID and bucket either way: Part 6 uses them to ship your model.

In [ ]:
USE_MY_OWN_DATA = False            # True = train on YOUR bucket's listings_master.csv
PROJECT_ID = "YOUR-PROJECT-ID"     # e.g. "myscrapers-dww05002-f26-510015"
BUCKET_NAME = "YOUR-PROJECT-ID"    # usually the same as your project ID

In [ ]:
import io
import json

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module3/data/lab4_listings_master_snapshot.csv"

if USE_MY_OWN_DATA:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    my_bucket = storage.Client(project=PROJECT_ID).bucket(BUCKET_NAME)
    csv_bytes = my_bucket.blob("structured/datasets/listings_master.csv").download_as_bytes()
    raw = pd.read_csv(io.BytesIO(csv_bytes), dtype={"post_id": str})
    print("Loaded YOUR listings_master.csv from gs://" + BUCKET_NAME)
    contact_rows = int((raw["make"].astype(str).str.lower() == "contact").sum())
    if contact_rows > 0:
        print("WARNING:", contact_rows, "cars still say make = 'contact'. Ship your A06 regex fix (and re-extract) first.")
    if len(raw) < 100:
        print("WARNING: only", len(raw), "cars. A custom model needs more data: let your pipeline run a few more days.")
else:
    raw = pd.read_csv(DATA_URL, dtype={"post_id": str})
    print("Loaded Dr. Wanik's cleaned Hartford snapshot")

print(raw.shape[0], "cars,", raw.shape[1], "columns")
raw.head()

# Part 2 · Clean (10 min)

The regex fixed `Contact Information`, but real data always has a little mess left. Look at the make counts:
`ford` and `Ford` are the same car, and so are `chevy`, `chevrolet` and `Chevrolet`. A model would treat them as
different brands.

**Important:** `predict-shap` runs **this exact cleaning** on every new car before predicting. If you change it
here, your model and production stop agreeing.

In [ ]:
print(raw["make"].value_counts().head(12))

In [ ]:
FEATURES = ["year", "mileage", "make", "model"]
NICKNAMES = {"chevy": "chevrolet", "vw": "volkswagen", "mercedes": "mercedes-benz", "benz": "mercedes-benz"}


def clean_cars(df):
    out = df.copy()
    for col in ["price", "year", "mileage"]:
        out[col] = pd.to_numeric(out[col].astype(str).str.replace(r"[^\d.]+", "", regex=True), errors="coerce")
    for col in ["make", "model"]:
        out[col] = out[col].astype(str).str.strip().str.lower()
        out.loc[out[col].isin(["", "nan", "none"]), col] = np.nan
    out["make"] = out["make"].replace(NICKNAMES)
    return out


cars = clean_cars(raw)
print(cars["make"].value_counts().head(10))

## 2b. Flag the rows a model should not learn from
Same idea as Lab 3's review flags: an odd price, a placeholder mileage (`999999` means "I don't know"), or a very old
classic car priced by a collector. Flagged rows stay in the data, but they do **not** train the model.

The flag is a small function, because you use it **twice**: here, and again in Part 7 on your production cars.

In [ ]:
def review_reasons(df, price_col):
    reasons = []
    for i in range(len(df)):
        row = df.iloc[i]
        why = []
        if pd.isna(row[price_col]) or row[price_col] < 500 or row[price_col] > 100000:
            why.append("odd price")
        if row["mileage"] in [0, 99999, 999999]:
            why.append("placeholder mileage")
        if pd.isna(row["year"]) or row["year"] < 1950:
            why.append("very old")
        reasons.append(", ".join(why))
    return reasons


cars["reason"] = review_reasons(cars, "price")
cars["needs_review"] = cars["reason"] != ""

model_ready = cars[cars["needs_review"] == False].copy()
print(len(model_ready), "of", len(cars), "cars are ready for a model")
cars[cars["needs_review"]][["post_id", "year", "make", "model", "price", "mileage", "reason"]]

# Part 3 · Baseline (10 min)

Before tuning anything, write down the numbers to beat:
1. **Guess the median price for every car.** If your model cannot beat this, it learned nothing.
2. **A plain decision tree** with the same settings as `train-dt`, the model your pipeline already trains every hour.

We hold out 20% of the cars as a **test set** and never tune on them.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

X = model_ready[FEATURES]
y = model_ready["price"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(len(X_train), "training cars |", len(X_test), "test cars")


def make_pipe(estimator):
    numeric = Pipeline([("imp", SimpleImputer(strategy="median")), ("scale", StandardScaler())])
    categorical = Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                            ("oh", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=2))])
    pre = ColumnTransformer([("num", numeric, ["year", "mileage"]), ("cat", categorical, ["make", "model"])])
    return Pipeline([("pre", pre), ("model", estimator)])


median_mae = mean_absolute_error(y_test, np.full(len(y_test), y_train.median()))
baseline = make_pipe(DecisionTreeRegressor(max_depth=12, min_samples_leaf=10, random_state=42))
baseline.fit(X_train, y_train)
baseline_mae = mean_absolute_error(y_test, baseline.predict(X_test))
baseline_cv_scores = cross_val_score(baseline, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
baseline_cv_mae = -baseline_cv_scores.mean()
print("Guess the median:         test MAE $" + format(round(median_mae), ","))
print("Baseline tree (train-dt): test MAE $" + format(round(baseline_mae), ","),
      "| cross-validation MAE $" + format(round(baseline_cv_mae), ","))

`make_pipe` is the whole recipe in one object: fill missing values, scale the numbers, one-hot encode make and model
(rare ones are grouped as "infrequent"), then the model. **You save and ship the whole pipeline**, so production
gets the same preprocessing for free.

The baseline has **two** scores. **Test MAE** is its error on the held-out test cars. **Cross-validation MAE** splits
the training cars into 5 parts, trains on 4, scores on the 5th, and repeats, so every training car gets a turn at
being "new". You tune with cross-validation and keep the test cars for the very end.

# Part 4 · Tune three ways (25 min)

## 4a. Guess first
Before you see any results, make two guesses. A wrong guess costs nothing: the notebook just tells you later
whether you called it.
- **Which model family wins?** A single decision tree, a random forest (many trees averaged), or KNN (price a car like
  its most similar neighbors)?
- **Which feature moves prices the most?** `year`, `mileage`, `make` or `model`?

In [ ]:
MY_GUESS_WINNER = "?"          # <-- "decision_tree", "random_forest" or "knn"
MY_GUESS_TOP_FEATURE = "?"     # <-- "year", "mileage", "make" or "model"

## 4b. Tune
`GridSearchCV` tries every combination in a grid, scores each one with **5-fold cross-validation** on the training
cars (MAE, lower is better), and keeps the best. Three model families, three grids. Run it: it takes a minute or two.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor

candidates = {
    "decision_tree": (DecisionTreeRegressor(random_state=42),
                      {"model__max_depth": [3, 5, 8, 12, None], "model__min_samples_leaf": [1, 5, 10, 20]}),
    "random_forest": (RandomForestRegressor(random_state=42, n_jobs=-1),
                      {"model__n_estimators": [100, 300], "model__max_depth": [None, 10],
                       "model__min_samples_leaf": [1, 3, 5]}),
    "knn": (KNeighborsRegressor(),
            {"model__n_neighbors": [3, 5, 7, 11, 15], "model__weights": ["uniform", "distance"]}),
}

searches = {}
rows = []
for name in candidates:
    estimator, grid = candidates[name]
    search = GridSearchCV(make_pipe(estimator), grid, cv=5, scoring="neg_mean_absolute_error", n_jobs=-1)
    search.fit(X_train, y_train)
    searches[name] = search
    test_mae = mean_absolute_error(y_test, search.best_estimator_.predict(X_test))
    rows.append({"model": name, "cv_mae": round(-search.best_score_), "test_mae": round(test_mae),
                 "best_params": search.best_params_})

rows.append({"model": "baseline tree (not tuned)", "cv_mae": round(baseline_cv_mae), "test_mae": round(baseline_mae),
             "best_params": "max_depth=12, min_samples_leaf=10"})
results = pd.DataFrame(rows).sort_values("cv_mae").reset_index(drop=True)
print("You guessed:", MY_GUESS_WINNER, "| lowest cross-validation MAE:", results.iloc[0]["model"])
results

## 4c. Pick your champion
Pick by **cross-validation MAE** (`cv_mae`), not test MAE: the test set is your final exam, so you do not choose with
it. Type the name below (`decision_tree`, `random_forest` or `knn`).

**Why do `cv_mae` and `test_mae` disagree?** The test set is only about 50 cars, so two or three unusual cars can move
its MAE by hundreds of dollars. Cross-validation averages five different splits, so it is the steadier number. A model
can even look worse than the baseline on one and better on the other. That is small data, not a bug.

In [ ]:
CHAMPION = "PICK_ONE"     # <-- your pick: "decision_tree", "random_forest" or "knn"

if CHAMPION not in searches:
    raise ValueError('Type your champion on the first line of this cell: "decision_tree", "random_forest" or "knn"')
champion = searches[CHAMPION].best_estimator_
champion_cv_mae = -searches[CHAMPION].best_score_
champion_test_mae = mean_absolute_error(y_test, champion.predict(X_test))
print("Champion:", CHAMPION, searches[CHAMPION].best_params_)
print("Champion: cross-validation MAE $" + format(round(champion_cv_mae), ","), "| test MAE $" + format(round(champion_test_mae), ","))
print("Baseline: cross-validation MAE $" + format(round(baseline_cv_mae), ","), "| test MAE $" + format(round(baseline_mae), ","))

# Part 5 · Explain it before you ship it (20 min)

Lab 2 came back. SHAP splits each prediction into **one contribution per feature**, measured from the average
prediction (the **base value**): `base value + SHAP(year) + SHAP(mileage) + SHAP(make) + SHAP(model) = prediction`.

The helper below is the **same code `predict-shap` runs in production**. Trees get exact, fast SHAP (`TreeExplainer`);
KNN gets a slower, sampled estimate (`KernelExplainer`). One-hot columns are added back up, so you see 4 features, not 150.

In [ ]:
def original_feature(transformed_name):
    name = transformed_name.split("__", 1)[-1]
    for col in ["make", "model"]:
        if name.startswith(col + "_"):
            return col
    for col in ["year", "mileage"]:
        if name == col or name.startswith(col):
            return col
    return name


def explain_rows(pipe, X, background):
    pre = pipe.named_steps["pre"]
    est = pipe.named_steps["model"]
    Xt = pre.transform(X)
    if hasattr(Xt, "toarray"):
        Xt = Xt.toarray()
    names = list(pre.get_feature_names_out())
    if hasattr(est, "tree_") or hasattr(est, "estimators_"):
        explainer = shap.TreeExplainer(est)
        values = explainer.shap_values(Xt, check_additivity=False)
        base = float(np.ravel(explainer.expected_value)[0])
    else:
        Bt = pre.transform(background)
        if hasattr(Bt, "toarray"):
            Bt = Bt.toarray()
        summary = shap.kmeans(Bt, min(10, len(Bt)))
        explainer = shap.KernelExplainer(est.predict, summary)
        values = explainer.shap_values(Xt, nsamples=100, silent=True)
        base = float(np.ravel(explainer.expected_value)[0])
    values = np.asarray(values)
    shap_columns = []
    for feature in FEATURES:
        shap_columns.append("shap_" + feature)
    grouped = pd.DataFrame(0.0, index=X.index, columns=shap_columns)
    for j in range(len(names)):
        grouped["shap_" + original_feature(names[j])] += values[:, j]
    return base, grouped


background = X_train.sample(min(50, len(X_train)), random_state=0)
base_value, test_shap = explain_rows(champion, X_test, background)
check = base_value + test_shap.sum(axis=1) - champion.predict(X_test)
print("Base value (average prediction): $" + format(round(base_value), ","))
print("Largest gap between base + SHAP and the prediction: $" + str(round(float(np.abs(check).max()), 2)))

**Which features drive your model's prices?** Mean absolute SHAP = how many dollars each feature moves a typical prediction.

In [ ]:
importance = test_shap.abs().mean().sort_values()
importance.index = importance.index.str.replace("shap_", "")
importance.plot(kind="barh", color="#6D28A3")
plt.xlabel("Mean |SHAP| (dollars)")
plt.title("What drives " + CHAMPION + "'s prices (test cars)")
plt.show()
print("You guessed:", MY_GUESS_TOP_FEATURE, "| top feature:", importance.index[-1])

**One car, explained.** The waterfall starts at the base value and adds each feature's push until it reaches the
prediction. Change `CAR` to look at another test car.

In [ ]:
def waterfall(base, shap_row, feature_row):
    explanation = shap.Explanation(values=np.array(shap_row, dtype=float), base_values=base,
                                   data=np.array(feature_row, dtype=object), feature_names=FEATURES)
    shap.plots.waterfall(explanation, show=True)


CAR = 0
car_features = X_test.iloc[CAR]
print("Actual price: $" + format(round(y_test.iloc[CAR]), ","), "| predicted: $" + format(round(champion.predict(X_test.iloc[[CAR]])[0]), ","))
waterfall(base_value, test_shap.iloc[CAR].values, car_features.values)

## 5b. Price a car you know
Your car, a parent's car, the car you wish you had. Type it in, and your champion prices it and explains itself.
Write the make and model the way Craigslist sellers do (`honda`, `civic`).

If the model has never seen that make or model, it treats it as "a rare car" and leans on year and mileage alone.
The cell tells you how many it trained on.

In [ ]:
MY_YEAR = 2015
MY_MILEAGE = 90000
MY_MAKE = "honda"
MY_MODEL = "civic"

my_car = pd.DataFrame([{"price": np.nan, "year": MY_YEAR, "mileage": MY_MILEAGE, "make": MY_MAKE, "model": MY_MODEL}])
my_car = clean_cars(my_car)[FEATURES]
my_price = champion.predict(my_car)[0]
my_base, my_shap = explain_rows(champion, my_car, background)
same_make = int((X_train["make"] == my_car.iloc[0]["make"]).sum())
same_model = int((X_train["model"] == my_car.iloc[0]["model"]).sum())
print("Your model trained on", same_make, "cars with make =", my_car.iloc[0]["make"],
      "and", same_model, "with model =", my_car.iloc[0]["model"])
print("Your model's price: $" + format(round(my_price), ","))
waterfall(my_base, my_shap.iloc[0].values, my_car.iloc[0].values)

# Part 6 · Ship it (20 min)

## 6a. Save the model and its model card
`model.joblib` is your whole pipeline, frozen. `model_card.json` is its label: what it is, how good it was, which
library versions made it, and which cars it trained on (so production can tell **new** cars from cars it has seen).

In [ ]:
from datetime import datetime, timezone

joblib.dump(champion, "/content/model.joblib")
best_params = {}
for key in searches[CHAMPION].best_params_:
    best_params[key] = str(searches[CHAMPION].best_params_[key])
card = {
    "model_type": type(champion.named_steps["model"]).__name__,
    "champion": CHAMPION,
    "best_params": best_params,
    "trained_at": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "sklearn_version": sklearn.__version__,
    "features": FEATURES,
    "cv_mae": round(float(champion_cv_mae), 2),
    "test_mae": round(float(champion_test_mae), 2),
    "baseline_mae": round(float(baseline_mae), 2),
    "train_post_ids": list(model_ready.loc[X_train.index, "post_id"].astype(str)),
}
with open("/content/model_card.json", "w") as f:
    json.dump(card, f, indent=2)
print("Saved /content/model.joblib and /content/model_card.json (see the Files panel on the left)")
card_to_show = dict(card)
card_to_show.pop("train_post_ids")
print(json.dumps(card_to_show, indent=2))

## 6b. Check that `predict-shap` is deployed
Every case (A, B or C) ends the same way: a Cloud Function called `predict-shap` that is **ACTIVE**. Check in Cloud Shell:
```bash
gcloud functions describe predict-shap --gen2 --region=us-central1 --format='value(state)'
```
`ACTIVE` means you are ready. An error that it does not exist: go back to Part 0 (Case A or B: look for a red X in the
Actions tab and paste the failed step's log into Claude; Case C: paste the Cloud Shell output into Claude).

## 6c. Upload your model to your bucket
This uses the `PROJECT_ID` and `BUCKET_NAME` you typed in Part 1b. Run it and pick your **Google Cloud Gmail** in the
pop-up. No Google Cloud yet? Set `UPLOAD = False`: you can still do Part 7 in demo mode.

In [ ]:
UPLOAD = True

if UPLOAD and ("YOUR-PROJECT-ID" in PROJECT_ID or "YOUR-PROJECT-ID" in BUCKET_NAME):
    print("Type your PROJECT_ID and BUCKET_NAME in Part 1b, re-run that cell, then re-run this one.")
    print("Staying in demo mode until you do.")
    UPLOAD = False

if UPLOAD:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    client = storage.Client(project=PROJECT_ID)
    bucket = client.bucket(BUCKET_NAME)
    bucket.blob("models/model.joblib").upload_from_filename("/content/model.joblib")
    bucket.blob("models/model_card.json").upload_from_filename("/content/model_card.json")
    print("Uploaded to gs://" + BUCKET_NAME + "/models/  -> predict-shap uses it at the next :25")

## 6d. Run it now (Cloud Shell)
Instead of waiting for :25, open **Cloud Shell** and run it yourself. The next cell prints the commands with **your**
project and bucket already filled in: copy them one at a time.

In [ ]:
print("1. Point Cloud Shell at your project, then run predict-shap now:")
print()
print("gcloud config set project " + PROJECT_ID)
print("gcloud scheduler jobs run predict-shap-hourly --location=us-central1")
print()
print("2. Wait about a minute, then look at what it wrote:")
print()
print('gcloud storage cat "gs://' + BUCKET_NAME + '/structured/preds_lab4/preds_master.csv" | head -5')
print('gcloud storage cat "gs://' + BUCKET_NAME + '/structured/preds_lab4/shap_master.csv" | head -5')
print()
print("3. Nothing there? Read its log and paste it into Claude:")
print()
print("gcloud functions logs read predict-shap --region=us-central1 --gen2 --limit=20")

# Part 7 · Read production (15 min)

Your pipeline now saves two files every hour, one row per car:
- `structured/preds_lab4/preds_master.csv`: the car, the **predicted** price, the actual asking price, the error,
  `seen_in_training` (did your model train on this car?), and which model version made the prediction
- `structured/preds_lab4/shap_master.csv`: the **SHAP values** behind each prediction, plus the base value

The next cell reads them from your bucket, for the model you uploaded. If production has not run yet, it tells you
what to do and switches to **demo mode**: the same two tables, built right here from Hartford cars, so the rest of
the notebook still runs.

In [ ]:
def read_bucket_csv(key):
    blob = bucket.blob(key)
    if not blob.exists():
        return None
    return pd.read_csv(io.BytesIO(blob.download_as_bytes()), dtype={"post_id": str, "model_version": str})


def demo_tables():
    seen_cars = cars.loc[X_train.index].sample(50, random_state=1)
    demo_cars = pd.concat([seen_cars, cars.loc[X_test.index], cars[cars["needs_review"]]])
    demo_preds = demo_cars[["post_id", "year", "mileage", "make", "model"]].copy()
    demo_preds["actual_price"] = demo_cars["price"]
    demo_preds["pred_price"] = champion.predict(demo_cars[FEATURES]).round(2)
    demo_preds["abs_error"] = (demo_preds["pred_price"] - demo_preds["actual_price"]).abs()
    demo_preds["seen_in_training"] = demo_cars.index.isin(X_train.index)
    demo_base, demo_shap = explain_rows(champion, demo_cars[FEATURES], background)
    demo_shap["post_id"] = demo_cars["post_id"]
    demo_shap["base_value"] = demo_base
    demo_shap["pred_price"] = demo_preds["pred_price"]
    return demo_preds, demo_shap


IN_PRODUCTION = False
if UPLOAD:
    live_card = json.loads(bucket.blob("models/model_card.json").download_as_text())
    live_version = str(live_card["trained_at"])
    preds = read_bucket_csv("structured/preds_lab4/preds_master.csv")
    prod_shap = read_bucket_csv("structured/preds_lab4/shap_master.csv")
    if preds is not None and prod_shap is not None:
        preds = preds[preds["model_version"] == live_version].copy()
        prod_shap = prod_shap[prod_shap["model_version"] == live_version].copy()
        if len(preds) > 0:
            IN_PRODUCTION = True
    if IN_PRODUCTION:
        print("PRODUCTION:", len(preds), "predictions from your", live_card["model_type"], "(version " + live_version + ")")
    else:
        print("No production predictions yet for the model you uploaded (version " + live_version + ").")
        print("Run the commands from 6d in Cloud Shell, wait a minute, then re-run this cell.")

if not IN_PRODUCTION:
    print("DEMO MODE: building the two production tables here, from Hartford cars")
    preds, prod_shap = demo_tables()

preds[["post_id", "year", "make", "model", "mileage", "actual_price", "pred_price", "abs_error", "seen_in_training"]].tail(10)

## 7b. Production predicts every car, even the junk ones
In Part 2 you kept three kinds of rows away from training. Production cannot be that picky: it has to answer for
whatever the scraper brings in. And it can, because **price is not an input**. A `$1` "call me" ad still has a year, a
make and a model, so it still gets a prediction.

What you must not do is **score** the model against a junk price. So production predicts everything, and you flag
before you score, with the same function as Part 2:

In [ ]:
preds["reason"] = review_reasons(preds, "actual_price")
preds["needs_review"] = preds["reason"] != ""
scored = preds[preds["needs_review"] == False].copy()
print(len(preds), "production predictions |", len(scored), "can be scored |", int(preds["needs_review"].sum()), "flagged")
preds[preds["needs_review"]][["post_id", "year", "make", "model", "mileage", "actual_price", "pred_price", "reason"]].head(10)

## 7c. Honest scoring
Up to three groups of cars, one error each:
- **Test cars (Part 4):** held out in this notebook, from the same data the model trained on.
- **Production, seen in training:** cars the model trained on. A forest or KNN partly memorizes them, so this error
  usually flatters the model. (A shallow tree memorizes very little, so its two production errors can be close.)
- **Production, not seen:** cars the model never trained on. This is what your model is really worth.

Tonight's model learned **Hartford** prices. If your pipeline scrapes a different area, every production car is
"not seen", **and** it comes from a different market.

In [ ]:
seen = scored[scored["seen_in_training"] == True]
unseen = scored[scored["seen_in_training"] == False]
score_rows = [{"cars": "test cars (Part 4)", "how many": len(X_test), "mean abs error ($)": round(champion_test_mae)}]
if len(seen) > 0:
    score_rows.append({"cars": "production, seen in training", "how many": len(seen),
                       "mean abs error ($)": round(seen["abs_error"].mean())})
if len(unseen) > 0:
    score_rows.append({"cars": "production, not seen", "how many": len(unseen),
                       "mean abs error ($)": round(unseen["abs_error"].mean())})
pd.DataFrame(score_rows)

## 7d. Explain a production prediction, and read the ad
Pick a car by row number (the last one is `-1`). You get why your model priced it that way, straight from the SHAP
values production saved, and then the ad itself, so you can judge with your own eyes.

In [ ]:
ad_index = None
if IN_PRODUCTION:
    ad_index = read_bucket_csv("structured/datasets/listings_master.csv")


def show_ad(post_id):
    if ad_index is None or "source_txt" not in ad_index.columns:
        print("(Demo mode: the ad text lives in your bucket. It shows here once you are reading real production.)")
        return
    match = ad_index[ad_index["post_id"] == post_id]
    if len(match) == 0 or pd.isna(match.iloc[-1]["source_txt"]):
        print("No saved ad text for", post_id)
        return
    blob = bucket.blob(match.iloc[-1]["source_txt"])
    if not blob.exists():
        print("No saved ad text for", post_id)
        return
    print("----- the ad (first 1,500 characters) -----")
    print(blob.download_as_text()[:1500])


ROW = -1
car_row = scored.iloc[ROW]
shap_row = prod_shap[prod_shap["post_id"] == car_row["post_id"]].iloc[-1]
print(car_row["year"], car_row["make"], car_row["model"], "|", car_row["mileage"], "miles")
print("Asking $" + format(round(car_row["actual_price"]), ","), "| your model says $" + format(round(car_row["pred_price"]), ","))
waterfall(float(shap_row["base_value"]),
          [shap_row["shap_year"], shap_row["shap_mileage"], shap_row["shap_make"], shap_row["shap_model"]],
          [car_row["year"], car_row["mileage"], car_row["make"], car_row["model"]])
show_ad(car_row["post_id"])

## 7e. Bargain, problem, or mistake?
A big negative gap means the seller is asking far less than your model expects. That is one of three things:
1. **A real bargain.**
2. **A problem the four features cannot see:** a rebuilt title, a blown transmission, "needs work".
3. **An extraction mistake.** Look at the `model` column. Dr. Wanik's Hartford data has a `chrysler town` and a
   `jeep grand`: the Lab 3 regex keeps one word after the make, so a Town & Country becomes `town` and a Grand
   Cherokee becomes `grand`. A model cannot price a car it cannot identify. (This is what A07 is for: Gemini reads
   the whole ad.)

In [ ]:
scored["gap"] = scored["actual_price"] - scored["pred_price"]
bargains = scored.sort_values("gap")[["post_id", "year", "make", "model", "mileage", "actual_price", "pred_price", "gap"]].head(5)
bargains

Read one. `PICK = 0` is the biggest gap; try `1` to `4` for the others. Which of the three is it?

In [ ]:
PICK = 0
show_ad(bargains.iloc[PICK]["post_id"])

# Part 8 · Wrap up (5 min)

## Four short answers (1 or 2 sentences each)
1. Which model family won on cross-validation MAE, and by how many dollars did it beat the baseline tree? Did you guess it?
2. The car you priced in 5b: was the price believable? Which feature pushed it the most, and in which direction?
3. Part 7c: compare your model's error on the test cars with its error in production. Are they close? Give one
   reason they could differ.
4. Part 7e: the "bargain" you read. Real bargain, hidden problem, or extraction mistake? How can you tell?
   (Demo mode: answer from the table.)

*Your answers:*

---

---

## Want a custom model on your own cars? (optional, any time after tonight)
Tonight everyone trained on Dr. Wanik's Hartford cars, so everyone succeeds. Your pipeline keeps collecting **your**
area's cars every hour, so later you can train a model that knows your market:
1. **Check your data first** (Cloud Shell). A06 fixed? Enough cars? Count the makes and the rows:
   ```bash
   gcloud storage cat "gs://YOUR-BUCKET/structured/datasets/listings_master.csv" | cut -d, -f6 | sort | uniq -c | sort -rn | head
   gcloud storage cat "gs://YOUR-BUCKET/structured/datasets/listings_master.csv" | wc -l
   ```
   No `contact` at the top and 100+ rows? You are ready.
2. **Re-run this notebook** with `USE_MY_OWN_DATA = True` in Part 1b. Everything else stays the same: clean, baseline,
   tune, explain, ship.
3. **Ship it** (Part 6c uploads the new `model.joblib` over the old one). `predict-shap` notices the new model version and
   re-predicts and re-explains all your cars with it. The old predictions stay in the files, tagged with the old version,
   so you can compare the two models. Now "seen in training" and "not seen" in Part 7c are both **your** cars.
4. Want a copy of your data on your laptop? `gcloud storage cp "gs://YOUR-BUCKET/structured/datasets/listings_master.csv" .`
   in Cloud Shell, then **⋮ → Download** (Cloud Shell's menu).

**Submit:** File → Download → `.ipynb`, then upload it to **Submit Lab 4** on HuskyCT.
**Leave `predict-shap` running.** Every hour it predicts and explains the newest cars, and your midterm can chart how
your model's errors and explanations move over time.